# ⚛️ QuantumShield — Quantum Kernel Network Intrusion Detection

**Interstate Quantum Hackathon Fall 2026 · QML Track · Two-person team**

**Research question:** Does a four-qubit *entangling* quantum feature map help distinguish benign from malicious network traffic, relative to a non-entangling map and a classical RBF SVM?

This notebook builds on the organizers' `QML_1_Lecture.ipynb`, particularly Parts 2–5. It uses **Qiskit Machine Learning** and a **local statevector simulator**, so no Google account or quantum computer is required.

**Plan:** Load NSL-KDD → preprocess 4 traffic features → tune quantum encoding bandwidth on training data → fit classical and quantum classifiers → measure honest held-out performance → generate report figures.

> **Scope & limitations:** A small, class-balanced subsample from a legacy benchmark is a *methodology demonstration*, not a working enterprise intrusion-detection system. No claim of quantum speedup is made. Simulator runtime is classical runtime.

In [ ]:
# Hackathon requirement: first command installs packages needed to run from a clean notebook.
# When already installed in your qml-env, this should finish quickly.
%pip install -q "qiskit>=2.0,<3" "qiskit-machine-learning>=0.9,<1" "scikit-learn>=1.5,<2" "numpy>=2,<3" "pandas>=2,<3" "matplotlib>=3.8,<4" "ipykernel"

## 1. Imports and reproducible experiment settings

We keep a fixed seed and start with **40 benign + 40 attack** training records, and **20 + 20** independent test records. This size is deliberate: a quantum kernel computes similarities for many pairs of samples.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay,
)
from qiskit.circuit.library import zz_feature_map, z_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms import QSVC

SEED = 42
TRAIN_PER_CLASS = 40
TEST_PER_CLASS = 20
REPS = 2
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print("QuantumShield environment ready.")

## 2. Download real labeled cybersecurity data

**Source:** [NSL-KDD GitHub mirror](https://github.com/Jehuty4949/NSL_KDD), containing public legacy intrusion-detection benchmark files.

- `KDDTrain+_20Percent.txt` is a training-only subset.
- `KDDTest+.txt` is a separate test split; we never use it to tune parameters.
- Each record has **41 features**, then an attack name (`normal` or a named attack), then a difficulty field.

The two files are downloaded on the first run and cached locally. **Internet access is needed only for this download.** If a download fails, open the URL displayed in the error message and save that file to the notebook's `data/` folder.

In [ ]:
DATA_URLS = {
    "KDDTrain+_20Percent.txt": "https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTrain%2B_20Percent.txt",
    "KDDTest+.txt": "https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTest%2B.txt",
}
for name, url in DATA_URLS.items():
    target = DATA_DIR / name
    if not target.exists():
        print(f"Downloading {name} ...")
        try:
            urlretrieve(url, target)
        except Exception as exc:
            raise RuntimeError(f"Could not download {name}. Manually download {url} into {DATA_DIR.resolve()}") from exc
    else:
        print(f"Found {target}")

raw_train = pd.read_csv(DATA_DIR / "KDDTrain+_20Percent.txt", header=None)
raw_test  = pd.read_csv(DATA_DIR / "KDDTest+.txt", header=None)
assert raw_train.shape[1] == raw_test.shape[1] == 43, "Expected 43 NSL-KDD fields"
print("Official source rows — train:", len(raw_train), "test:", len(raw_test))
print("Example attack labels:", raw_train.iloc[:,41].value_counts().head().to_dict())

## 3. Select four network features and balanced samples

Our four inputs are numerical fields with known positions in NSL-KDD:

| Input | NSL-KDD column (zero-based) | Meaning |
|---|---:|---|
| `src_bytes` | 4 | Bytes from source to destination |
| `dst_bytes` | 5 | Bytes from destination to source |
| `count` | 22 | Recent connections to the same destination host |
| `srv_serror_rate` | 25 | SYN-error rate for the same service |

The true label is in column **41**: `normal` means benign; everything else is an attack. The difficulty field in column 42 is **never used** as a predictor.

We apply `log1p` to skewed count/byte columns, then fit MinMax scaling **only on training examples**. This avoids leaking test information into the model.

In [ ]:
FEATURE_COLUMNS = {"src_bytes": 4, "dst_bytes": 5, "count": 22, "srv_serror_rate": 25}
FEATURE_NAMES = list(FEATURE_COLUMNS)

def prepare_frame(raw):
    out = pd.DataFrame({name: pd.to_numeric(raw.iloc[:, idx], errors="raise")
                        for name, idx in FEATURE_COLUMNS.items()})
    out["is_attack"] = (raw.iloc[:, 41].astype(str).str.strip() != "normal").astype(int).to_numpy()
    return out.dropna().reset_index(drop=True)

def balanced_subset(df, per_class, seed):
    pieces = []
    for label in (0, 1):
        subset = df.loc[df["is_attack"] == label]
        if len(subset) < per_class:
            raise ValueError(f"Not enough class-{label} records: {len(subset)}")
        pieces.append(subset.sample(n=per_class, random_state=seed))
    return pd.concat(pieces, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

train_df = balanced_subset(prepare_frame(raw_train), TRAIN_PER_CLASS, SEED)
test_df  = balanced_subset(prepare_frame(raw_test), TEST_PER_CLASS, SEED + 1)

X_train_raw = train_df[FEATURE_NAMES].to_numpy(dtype=float)
X_test_raw  = test_df[FEATURE_NAMES].to_numpy(dtype=float)
y_train = train_df["is_attack"].to_numpy()
y_test  = test_df["is_attack"].to_numpy()

# Log-scale the highly skewed nonnegative features; keep the error rate unchanged.
for index in (0, 1, 2):
    X_train_raw[:, index] = np.log1p(np.clip(X_train_raw[:, index], 0, None))
    X_test_raw[:, index] = np.log1p(np.clip(X_test_raw[:, index], 0, None))

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train_raw)
X_test = np.clip(scaler.transform(X_test_raw), 0, np.pi)

print("Train shape:", X_train.shape, "/ labels:", np.bincount(y_train))
print("Test shape:", X_test.shape, "/ labels:", np.bincount(y_test))
print("Sample encoded data [0, pi]:", np.round(X_train[0], 3))
assert X_train.shape[1] == 4 and X_test.shape[1] == 4
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()

## 4. Train-only tuning: quantum bandwidth λ

The lecture warns that quantum kernel values can become too small as we increase encoding strength. A **bandwidth** parameter λ scales the angles before the quantum circuit.

We compare λ values using **3-fold cross-validation on training data only**. We do *not* look at the held-out test set to choose λ. The four-qubit circuit uses a ZZ entangling feature map.

In [ ]:
LAMBDA_CANDIDATES = (0.25, 0.50, 1.0)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv_rows = []
start = time.perf_counter()

for lam in LAMBDA_CANDIDATES:
    fmap = zz_feature_map(feature_dimension=4, reps=REPS, entanglement="linear")
    kernel = FidelityStatevectorKernel(feature_map=fmap)
    # Exactly computed statevector kernel; train-only similarity matrix.
    K = kernel.evaluate(X_train * lam)
    fold_scores = []
    for train_idx, valid_idx in cv.split(X_train, y_train):
        svc = SVC(kernel="precomputed", C=1.0)
        svc.fit(K[np.ix_(train_idx, train_idx)], y_train[train_idx])
        predictions = svc.predict(K[np.ix_(valid_idx, train_idx)])
        fold_scores.append(balanced_accuracy_score(y_train[valid_idx], predictions))
    cv_rows.append({"lambda": lam, "cv_balanced_accuracy": float(np.mean(fold_scores)),
                    "cv_std": float(np.std(fold_scores))})

cv_results = pd.DataFrame(cv_rows)
display(cv_results)
best_lambda = float(cv_results.sort_values(
    ["cv_balanced_accuracy", "lambda"], ascending=[False, True]).iloc[0]["lambda"])
print(f"Selected lambda: {best_lambda} | CV time: {time.perf_counter()-start:.1f}s")
cv_results.to_csv(RESULTS_DIR / "bandwidth_cv.csv", index=False)

## 5. Train three classifiers — same records, same labels

- **Classical RBF SVM:** strong conventional nonlinear baseline.
- **Quantum ZZ QSVC:** includes two-qubit entangling interactions.
- **Quantum Z QSVC:** uses phase-encoding without entangling gates.

All three use `C=1.0` and the same training/test records. The quantum models use a **Qiskit statevector simulator** (no real QPU). This experiment compares representations, not computational acceleration.

In [ ]:
models = {}
preds = {}
fit_seconds = {}

# 1. Classical RBF baseline
start = time.perf_counter()
models["Classical RBF SVM"] = SVC(kernel="rbf", C=1.0, gamma="scale").fit(X_train, y_train)
fit_seconds["Classical RBF SVM"] = time.perf_counter() - start
preds["Classical RBF SVM"] = models["Classical RBF SVM"].predict(X_test)

# 2. Entangling quantum kernel SVM
fmap_zz = zz_feature_map(feature_dimension=4, reps=REPS, entanglement="linear")
qkernel_zz = FidelityStatevectorKernel(feature_map=fmap_zz)
start = time.perf_counter()
models["Quantum ZZ QSVC"] = QSVC(quantum_kernel=qkernel_zz, C=1.0).fit(X_train * best_lambda, y_train)
fit_seconds["Quantum ZZ QSVC"] = time.perf_counter() - start
preds["Quantum ZZ QSVC"] = models["Quantum ZZ QSVC"].predict(X_test * best_lambda)

# 3. Non-entangling quantum kernel SVM
fmap_z = z_feature_map(feature_dimension=4, reps=REPS)
qkernel_z = FidelityStatevectorKernel(feature_map=fmap_z)
start = time.perf_counter()
models["Quantum Z QSVC"] = QSVC(quantum_kernel=qkernel_z, C=1.0).fit(X_train * best_lambda, y_train)
fit_seconds["Quantum Z QSVC"] = time.perf_counter() - start
preds["Quantum Z QSVC"] = models["Quantum Z QSVC"].predict(X_test * best_lambda)

print("All three classifiers trained and evaluated on the held-out test subset.")

## 6. Results: do not assume a quantum win

**Attack = positive class (1).** For an intrusion detector, pay particular attention to attack recall (false negatives) and the false-positive rate (benign connections incorrectly flagged).

Because the test set has only 40 examples, **these results will be noisy**. Report that limitation instead of claiming deployment-ready accuracy.

In [ ]:
rows = []
for name, y_pred in preds.items():
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_attack": precision_score(y_test, y_pred, zero_division=0),
        "recall_attack": recall_score(y_test, y_pred, zero_division=0),
        "f1_attack": f1_score(y_test, y_pred, zero_division=0),
        "false_positive_rate": fp / (fp + tn),
        "false_negatives": int(fn),
        "fit_seconds": fit_seconds[name],
    })
metrics = pd.DataFrame(rows)
display(metrics.round(3))
metrics.to_csv(RESULTS_DIR / "metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(9,4))
xs = np.arange(len(metrics))
ax.bar(xs - .18, metrics["balanced_accuracy"], width=.35, label="Balanced accuracy")
ax.bar(xs + .18, metrics["f1_attack"], width=.35, label="Attack F1")
ax.set_xticks(xs, metrics["model"], rotation=10)
ax.set_ylabel("Held-out score (0–1)")
ax.set_ylim(0, 1.05)
ax.set_title("NSL-KDD: classical vs. quantum kernel classifiers")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "model_comparison.png", dpi=170)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12,3.7))
for ax, (name, y_pred) in zip(axes, preds.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, labels=[0,1],
        display_labels=["Benign", "Attack"],
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
fig.suptitle("Held-out confusion matrices", y=1.03)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrices.png", dpi=170, bbox_inches="tight")
plt.show()

## 7. Visualize the quantum representation

The kernel matrix entry $K_{ij}=|\langle\phi(x_i)|\phi(x_j)\rangle|^2$ measures the overlap of encoded states. We plot matrices sorted by the **training labels**. If the quantum representation is useful, we may see stronger within-class similarity than between-class similarity; this pattern is not guaranteed.

In [ ]:
order = np.argsort(y_train, kind="stable")
K_zz = qkernel_zz.evaluate(X_train * best_lambda)
K_z = qkernel_z.evaluate(X_train * best_lambda)

fig, axes = plt.subplots(1,2,figsize=(10,4.3))
for ax, K, title in zip(axes, (K_z, K_zz),
                        ("Z feature map (no entanglement)", "ZZ feature map (entangling)")):
    im = ax.imshow(K[np.ix_(order, order)], vmin=0, vmax=1, cmap="Blues", interpolation="nearest")
    ax.set_xlabel("Training observation, sorted by class")
    ax.set_ylabel("Training observation, sorted by class")
    ax.set_title(title)
    ax.axhline(TRAIN_PER_CLASS - .5, color="orange", lw=1)
    ax.axvline(TRAIN_PER_CLASS - .5, color="orange", lw=1)
fig.colorbar(im, ax=axes, label="Quantum-state overlap", shrink=.77)
fig.savefig(RESULTS_DIR / "kernel_comparison.png", dpi=170, bbox_inches="tight")
plt.show()
print("ZZ circuit:\n", fmap_zz.draw(output="text"))

## 8. Demonstration: classify held-out historical observations

This is a safe replay of labeled historical data, **not** live packet capture. It does not generate, launch, or block network attacks.

In [ ]:
demo = pd.DataFrame({
    "record": np.arange(1, 9),
    "actual": np.where(y_test[:8] == 1, "Attack", "Benign"),
    "classical": np.where(preds["Classical RBF SVM"][:8] == 1, "Attack", "Benign"),
    "quantum_ZZ": np.where(preds["Quantum ZZ QSVC"][:8] == 1, "Attack", "Benign"),
})
display(demo)
demo.to_csv(RESULTS_DIR / "demo_predictions.csv", index=False)

## 9. Save reproducibility information and write your report

All results are saved in `results/`. Include these figures and the `metrics.csv` table in your report. You must state:

1. Your hypothesis and chosen four features.
2. The exact sample size, class balancing, random seed, and source dataset.
3. The classical SVM baseline, Z and ZZ quantum feature maps, and λ cross-validation.
4. Your results **whether or not quantum wins**.
5. Limitations: tiny balanced subset, legacy data, four-feature reduction, simulator-only execution, no proven quantum advantage.

**Suggested extension only if time permits:** compare `reps=1` versus `reps=2`, keeping the test set untouched while selecting settings, or repeat the complete procedure with multiple preset random seeds and report the mean and variability.

In [ ]:
config = {
    "name": "QuantumShield — Interstate Quantum Hackathon 2026",
    "dataset": "NSL-KDD (public GitHub mirror)",
    "train_source": DATA_URLS["KDDTrain+_20Percent.txt"],
    "test_source": DATA_URLS["KDDTest+.txt"],
    "train_per_class": TRAIN_PER_CLASS,
    "test_per_class": TEST_PER_CLASS,
    "features": FEATURE_NAMES,
    "seed": SEED,
    "qubits": 4,
    "reps": REPS,
    "cv_folds": 3,
    "cv_lambda_candidates": list(LAMBDA_CANDIDATES),
    "selected_lambda": best_lambda,
    "models": list(models),
    "hardware": "classical statevector simulator — no quantum hardware",
}
(RESULTS_DIR / "experiment_config.json").write_text(json.dumps(config, indent=2))
print("Saved files:")
for p in sorted(RESULTS_DIR.glob("*")):
    print(" -", p)
print("\nNext: add the figures and metrics to PROJECT_REPORT.md, then commit your files to GitHub.")